In [2]:
# ==========================================
# BÀI 6: CHUYỂN ĐỔI KIỂU DỮ LIỆU (ĐỘC LẬP)
# ==========================================
import pandas as pd
import numpy as np
import re
from google.colab import drive

# 1. Mount Drive và Khởi tạo df_clean từ file thô
drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/Nhap_Mon_Hoc_May/Buoi2/Du_lieu_thuc_hanh_chuan_bi_du_lieu.xlsx'

df_raw = pd.read_excel(file_path, sheet_name='Du_lieu_tho')
df_clean = df_raw.copy()

print("=== BẮT ĐẦU THỰC HIỆN BÀI 6 ===\n")

# ------------------------------------------
# Yêu cầu 25: Price_VND - Chuyển về số VND, xử lý "miễn phí" và giá âm
# ------------------------------------------
def clean_price(val):
    if pd.isna(val):
        return np.nan
    val_str = str(val).lower().strip()

    # Xử lý "miễn phí" hoặc "free" -> 0
    if 'miễn phí' in val_str or 'free' in val_str:
        return 0.0

    # Loại bỏ đơn vị 'đ', 'vnd', khoảng trắng
    val_str = re.sub(r'[đ|vnd|\s]', '', val_str)

    # Loại bỏ dấu chấm phân cách hàng nghìn (350.000 -> 350000)
    val_str = val_str.replace('.', '')

    try:
        price = float(val_str)
        return price if price >= 0 else np.nan
    except:
        return np.nan

df_clean['Price_VND'] = df_clean['Price_VND'].apply(clean_price)
print("-> Đã chuyển đổi cột Price_VND")


# ------------------------------------------
# Yêu cầu 26: Rating - Tách phần số, chuyển float
# ------------------------------------------
def clean_rating(val):
    if pd.isna(val):
        return np.nan
    val_str = str(val).strip().replace(',', '.')

    match = re.search(r'\d+(\.\d+)?', val_str)
    if match:
        rating = float(match.group())
        return rating if 1.0 <= rating <= 5.0 else np.nan
    return np.nan

df_clean['Rating'] = df_clean['Rating'].apply(clean_rating)
print("-> Đã chuyển đổi cột Rating")


# ------------------------------------------
# Yêu cầu 27: Purchase_Count - Chuyển "1.2k" thành 1200 và "850 lượt" thành 850
# ------------------------------------------
def clean_purchase_count(val):
    if pd.isna(val):
        return np.nan
    val_str = str(val).lower().strip().replace(',', '.')
    val_str = val_str.replace('lượt', '').strip()

    if 'k' in val_str:
        num_str = val_str.replace('k', '').strip()
        try:
            return float(num_str) * 1000
        except:
            return np.nan

    try:
        return float(val_str)
    except:
        return np.nan

df_clean['Purchase_Count'] = df_clean['Purchase_Count'].apply(clean_purchase_count)
print("-> Đã chuyển đổi cột Purchase_Count")


# ------------------------------------------
# Yêu cầu 28: Weight - Quy đổi toàn bộ về gram
# ------------------------------------------
def clean_weight(val):
    if pd.isna(val):
        return np.nan
    val_str = str(val).lower().strip().replace(',', '.')

    if 'kg' in val_str:
        num_str = re.sub(r'[^\d.]', '', val_str)
        try:
            return float(num_str) * 1000
        except:
            return np.nan
    elif 'g' in val_str:
        num_str = re.sub(r'[^\d.]', '', val_str)
        try:
            return float(num_str)
        except:
            return np.nan
    else:
        try:
            return float(val_str)
        except:
            return np.nan

df_clean['Weight'] = df_clean['Weight'].apply(clean_weight)
print("-> Đã chuyển đổi cột Weight (về gram)")


# ------------------------------------------
# Yêu cầu 29: Discount - Chuyển phần trăm về tỷ lệ 0–1
# ------------------------------------------
def clean_discount(val):
    if pd.isna(val):
        return np.nan
    val_str = str(val).strip().replace(',', '.')

    if '%' in val_str:
        num_str = val_str.replace('%', '').strip()
        try:
            return float(num_str) / 100.0
        except:
            return np.nan

    try:
        disc = float(val_str)
        if disc > 1.0:
            disc = disc / 100.0
        return disc if 0.0 <= disc <= 1.0 else np.nan
    except:
        return np.nan

df_clean['Discount'] = df_clean['Discount'].apply(clean_discount)
print("-> Đã chuyển đổi cột Discount")


# ------------------------------------------
# Yêu cầu 30: Created_Date - Chuyển datetime, giá trị lỗi thành NaT
# ------------------------------------------
df_clean['Created_Date'] = pd.to_datetime(df_clean['Created_Date'], errors='coerce')
print("-> Đã chuyển đổi cột Created_Date sang kiểu datetime\n")


# ------------------------------------------
# Kiểm tra kết quả
# ------------------------------------------
print("--- TÓM TẮT KIỂU DỮ LIỆU SAU KHI CHUYỂN ĐỔI ---")
print(df_clean[['Price_VND', 'Rating', 'Purchase_Count', 'Weight', 'Discount', 'Created_Date']].dtypes)
display(df_clean[['Price_VND', 'Rating', 'Purchase_Count', 'Weight', 'Discount', 'Created_Date']].head())

Mounted at /content/drive
=== BẮT ĐẦU THỰC HIỆN BÀI 6 ===

-> Đã chuyển đổi cột Price_VND
-> Đã chuyển đổi cột Rating
-> Đã chuyển đổi cột Purchase_Count
-> Đã chuyển đổi cột Weight (về gram)
-> Đã chuyển đổi cột Discount
-> Đã chuyển đổi cột Created_Date sang kiểu datetime

--- TÓM TẮT KIỂU DỮ LIỆU SAU KHI CHUYỂN ĐỔI ---
Price_VND                float64
Rating                   float64
Purchase_Count           float64
Weight                   float64
Discount                 float64
Created_Date      datetime64[ns]
dtype: object


,Price_VND,Rating,Purchase_Count,Weight,Discount,Created_Date
0,625000.0,3.2,2290.0,1509.0,0.00,2026-07-17
1,377000.0,3.9,1405.0,754.0,0.00,2026-05-26
2,1039000.0,3.6,1081.0,989.0,0.25,2026-05-06
3,824000.0,4.4,323.0,416.0,0.15,2026-08-18
4,NaN,3.9,1655.0,137.0,0.25,2026-04-11
